# 영국 전력수요 데이터 수집·전처리·탐색적 분석

NESO가 공개한 Great Britain의 2022~2025년 30분 전력수요 자료를 사용한다. 이 노트북은 보고서 3~5절의 수집 확인, 시간축 전처리, 탐색적 분석을 재현한다.

JupyterLab에서 **Kernel → Restart Kernel and Run All Cells**를 선택하면 중간 표와 차트를 셀 아래에서 확인할 수 있다. Python 3.10 이상과 현재 Jupyter 환경의 IPython을 사용한다.

- 2022~2023년: 학습 자료 및 EDA
- 2024년: 검증 자료
- 2025년: 최종 평가 자료

윈도우 작업 폴더에 보관한 원본을 WSL에서 읽는다. 경로를 바꾸려면 첫 셀의 `DATA_DIR`을 수정한다. 수요의 단위는 MW이다.

## 1. 데이터 출처와 파일 확인

[NESO Historic Demand Data](https://www.neso.energy/data-portal/historic-demand-data)의 연도별 CSV를 사용한다. 수집 시 기록한 SHA-256과 대조하여 동일한 원본인지 확인한다. CSV가 다른 곳에 있다면 `DATA_DIR`을 수정한다. 원본이 없을 때 공식 주소에서 수집을 시도하려면 `DOWNLOAD_MISSING`을 `True`로 바꾼다. 공급자가 파일을 수정한 경우에는 기존 수집본을 사용해야 동일한 결과를 재현할 수 있다.

In [ ]:
import calendar
import csv
import hashlib
import io
import math
import os
import statistics as stats
import urllib.request
from collections import Counter, defaultdict
from datetime import date, datetime, time, timedelta, timezone
from functools import lru_cache
from html import escape
from pathlib import Path
from zoneinfo import ZoneInfo

from IPython.display import HTML, SVG, display

DATA_DIR = Path(os.environ.get('NESO_RAW_DIR',
    '/mnt/c/Users/임성훈/Documents/GitHub/NLPpj/gb-electricity-lstm/data/raw'))
if os.name == 'nt' and 'NESO_RAW_DIR' not in os.environ:
    DATA_DIR = Path('C:/Users/임성훈/Documents/GitHub/NLPpj/gb-electricity-lstm/data/raw')
DOWNLOAD_MISSING = False
SAVE_PROCESSED = False
OUTPUT_DIR = Path.cwd() / 'neso_notebook_outputs'
ROOT = RAW = DATA_DIR


def show_table(headers, rows):
    """Display the computed rows as a readable table in JupyterLab."""
    header = ''.join(f'<th>{escape(str(column))}</th>' for column in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(cell))}</td>' for cell in row) + '</tr>'
                   for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


print('원본 위치:', DATA_DIR)

In [ ]:
SNAPSHOT = {
    2022: ('https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bb44a1b5-75b1-4db2-8491-257f23385006/download/demanddata_2022.csv', 'fc6bd5d5a8e8b9094ae748ae6f62d688797577ca3be8e66acbffdda771a5b6c4'),
    2023: ('https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bf5ab335-9b40-4ea4-b93a-ab4af7bce003/download/demanddata_2023.csv', 'cb6f7b91dd6a5defbdd918e3db440b8e6cec5855ef18a8715bb8e294b50cfa23'),
    2024: ('https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/f6d02c0f-957b-48cb-82ee-09003f2ba759/download/demanddata_2024.csv', 'fa7895b6e9ab5eb1b450949d969cb18ffeb6005359fc71c79c87e1c47e59cac6'),
    2025: ('https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b2bde559-3455-4021-b179-dfe60c0337b0/download/demanddata_2025.csv', '8b8be946bbb084bc28fbf95a71fad0fceb5543b98e02755235130a42beaee4e0'),
}


def ensure_snapshot(year):
    url, expected_hash = SNAPSHOT[year]
    path = RAW / f'demanddata_{year}.csv'
    if path.exists():
        return path
    if not DOWNLOAD_MISSING:
        raise FileNotFoundError(f'{path}를 찾을 수 없습니다. DATA_DIR 또는 DOWNLOAD_MISSING을 확인하세요.')
    with urllib.request.urlopen(url, timeout=90) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != expected_hash:
        raise ValueError(f'{year}: 공식 서버 파일이 기존 수집본과 다릅니다.')
    RAW.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.csv.part')
    temporary.write_bytes(payload)
    temporary.replace(path)
    return path

## 2. 원본 구조와 품질 검사

연도별 날짜 형식을 해석하고, 모든 컬럼의 결측 표시와 `(정산 날짜, 정산 구간)` 중복을 확인한다. 봄·가을 시계 변경일에는 각각 46개·50개의 30분 구간이 있어, 이를 일별 기대값에 반영한다.

In [ ]:
def parse_date(value):
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d-%b-%Y', '%d-%b-%y', '%Y-%m-%dT%H:%M:%S'):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            pass
    raise ValueError(f'Unknown settlement date: {value!r}')


def expected_periods(day):
    # GB settlement days have 46/50 periods at the spring/autumn clock changes.
    for month, periods in ((3, 46), (10, 50)):
        last = date(day.year, month, calendar.monthrange(day.year, month)[1])
        last_sunday = last - timedelta(days=(last.weekday() + 1) % 7)
        if day == last_sunday:
            return periods
    return 48


def audit(path, year):
    payload = path.read_bytes()
    reader = csv.DictReader(io.StringIO(payload.decode('utf-8-sig'), newline=''))
    columns = reader.fieldnames
    if not columns or not {'SETTLEMENT_DATE', 'SETTLEMENT_PERIOD', 'ND'} <= set(columns):
        raise ValueError(f'{path.name}: required columns missing; got {columns}')
    missing = Counter({column: 0 for column in columns})
    keys = Counter()
    per_day = Counter()
    periods_by_day = {}
    values = []
    invalid_nd = []
    invalid_rows = []
    for line, row in enumerate(reader, start=2):
        if None in row:
            raise ValueError(f'{path.name}:{line}: extra CSV fields')
        for column in columns:
            if row[column] is None or row[column].strip().lower() in ('', 'na', 'nan', 'null', 'n/a'):
                missing[column] += 1
        try:
            day = parse_date(row['SETTLEMENT_DATE'].strip())
            period = int(row['SETTLEMENT_PERIOD'])
        except (ValueError, AttributeError, TypeError) as error:
            invalid_rows.append({'line': line, 'error': str(error)})
            continue
        keys[(day, period)] += 1
        per_day[day] += 1
        periods_by_day.setdefault(day, set()).add(period)
        try:
            value = float(row['ND'])
            if not math.isfinite(value):
                raise ValueError('non-finite ND')
            values.append(value)
        except (ValueError, TypeError):
            invalid_nd.append(line)
    start, end = date(year, 1, 1), date(year, 12, 31)
    expected_days = {start + timedelta(days=i) for i in range((end-start).days + 1)}
    period_issues = [str(day) for day in sorted(per_day)
                     if per_day[day] != expected_periods(day)
                     or periods_by_day[day] != set(range(1, expected_periods(day) + 1))]
    duplicates = sum(count - 1 for count in keys.values())
    missing_days = sorted(str(day) for day in expected_days - set(per_day))
    extra_days = sorted(str(day) for day in set(per_day) - expected_days)
    passed = not (duplicates or missing_days or extra_days or period_issues or invalid_nd or invalid_rows
                  or any(value <= 0 for value in values))
    return {
        'year': year, 'file': path.relative_to(ROOT).as_posix(),
        'bytes': len(payload), 'sha256': hashlib.sha256(payload).hexdigest(),
        'rows': sum(per_day.values()) + len(invalid_rows), 'columns': columns,
        'first_settlement_date': str(min(per_day)) if per_day else None,
        'last_settlement_date': str(max(per_day)) if per_day else None,
        'unique_days': len(per_day), 'missing_by_column': dict(missing),
        'duplicate_date_period_rows': duplicates, 'missing_days': missing_days,
        'out_of_year_days': extra_days, 'invalid_rows': invalid_rows,
        'invalid_nd_lines': invalid_nd, 'nonpositive_nd_count': sum(v <= 0 for v in values),
        'nd_min_mw': min(values) if values else None, 'nd_max_mw': max(values) if values else None,
        'period_count_distribution': dict(sorted(Counter(per_day.values()).items())),
        'clock_change_days': {str(day): count for day, count in sorted(per_day.items()) if count != 48},
        'unexpected_period_days': period_issues, 'structural_checks_passed': passed,
    }

In [ ]:
checks = {}
for year, (_, expected_hash) in SNAPSHOT.items():
    result = audit(ensure_snapshot(year), year)
    if result['sha256'] != expected_hash or not result['structural_checks_passed']:
        raise ValueError(f'{year}년 원본 해시 또는 데이터 구조 검사에 실패했습니다.')
    checks[year] = result

show_table(['연도', '시작일', '종료일', '행 수', '열 수', '결측 표시', '정산 키 중복'], [
    (year, r['first_settlement_date'], r['last_settlement_date'], f"{r['rows']:,}",
     len(r['columns']), sum(r['missing_by_column'].values()), r['duplicate_date_period_rows'])
    for year, r in checks.items()
])

# 컬럼별 결측 수를 직접 열어 볼 수 있다.
missing_by_year = {year: r['missing_by_column'] for year, r in checks.items()}

## 3. 30분 시간축 전처리

현지 자정을 UTC로 변환한 다음 `(구간 번호 - 1) × 30분`을 더한다. 계산한 UTC 시각을 영국 현지 시각으로 되돌려 시간대와 요일을 구한다. 비교를 위해 시간대 정보 없이 자정에 구간을 더했을 때의 중복과 누락도 계산한다.

In [ ]:
LONDON = ZoneInfo('Europe/London')
STEP = timedelta(minutes=30)
SPLITS = {2022: 'train', 2023: 'train', 2024: 'validation', 2025: 'test'}
FIELDS = ('settlement_date', 'settlement_period', 'timestamp_utc', 'timestamp_local',
          'nd_mw', 'local_half_hour', 'day_of_week', 'month', 'is_weekend')


@lru_cache(maxsize=None)
def utc_midnight(day):
    return datetime.combine(day, time(), tzinfo=LONDON).astimezone(timezone.utc)


def prepare_year(year):
    path = RAW / f'demanddata_{year}.csv'
    rows, naive_times = [], []
    with path.open(encoding='utf-8-sig', newline='') as stream:
        for raw in csv.DictReader(stream):
            day = parse_date(raw['SETTLEMENT_DATE'].strip())
            period = int(raw['SETTLEMENT_PERIOD'])
            offset = (period - 1) * STEP
            instant = utc_midnight(day) + offset
            local = instant.astimezone(LONDON)
            if local.date() != day:
                raise ValueError(f'{year}: invalid settlement period {day}, {period}')
            naive_times.append(datetime.combine(day, time()) + offset)
            rows.append(dict(zip(FIELDS, (
                day.isoformat(), period, instant.isoformat(), local.isoformat(),
                float(raw['ND']), local.hour * 2 + local.minute // 30,
                local.weekday(), local.month, int(local.weekday() >= 5),
            ))))
    rows.sort(key=lambda row: row['timestamp_utc'])
    naive_counter = Counter(naive_times)
    expected = {min(naive_times) + i * STEP for i in range(len(rows))}
    return rows, {
        'naive_duplicate_slots': sum(count - 1 for count in naive_counter.values()),
        'naive_missing_slots': len(expected - naive_counter.keys()),
    }

In [ ]:
partitions = {'train': [], 'validation': [], 'test': []}
conversion_checks = []
for year, split in SPLITS.items():
    rows, diagnostic = prepare_year(year)
    if len(rows) != checks[year]['rows']:
        raise ValueError(f'{year}년 전처리 전후의 행 수가 다릅니다.')
    partitions[split].extend(rows)
    conversion_checks.append((year, split, len(rows), diagnostic['naive_duplicate_slots'],
                              diagnostic['naive_missing_slots']))

all_rows = [row for rows in partitions.values() for row in rows]
instants = [datetime.fromisoformat(row['timestamp_utc']) for row in all_rows]
if not all(b - a == STEP for a, b in zip(instants, instants[1:])):
    raise ValueError('UTC 시간축에 30분이 아닌 간격 또는 중복이 있습니다.')

show_table(['연도', '분할', '처리 행 수', '단순 변환 시 중복', '단순 변환 시 누락'], conversion_checks)
print(f'수정한 UTC 시간축: {len(all_rows):,}행, 인접한 {len(instants)-1:,}개 간격이 모두 30분')
print('관측 범위:', instants[0].isoformat(), '~', instants[-1].isoformat())

examples = [row for row in partitions['train']
            if row['settlement_date'] in ('2023-03-26', '2023-10-29')
            and row['settlement_period'] <= 6]
show_table(['정산일', '구간 번호', 'UTC', '영국 현지 시각', '수요(MW)'],
           ([row[k] for k in FIELDS[:5]] for row in examples))

In [ ]:
show_table(['구간', '행 수', '열 수'], ((name, f'{len(rows):,}', len(FIELDS))
                                       for name, rows in partitions.items()))
show_table(FIELDS, ([row[column] for column in FIELDS] for row in partitions['train'][:5]))

if SAVE_PROCESSED:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, rows in partitions.items():
        with (OUTPUT_DIR / f'{name}.csv').open('w', encoding='utf-8', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=FIELDS)
            writer.writeheader()
            writer.writerows(rows)
    print('CSV 저장 위치:', OUTPUT_DIR)

## 4. 탐색적 분석

여기부터는 2022~2023년 학습 관측만 사용한다. 표에 쓰인 숫자와 차트는 위 셀에서 만든 전처리 자료로 다시 계산한다. 요일은 영국 현지 시각 기준이며, 주말은 토요일·일요일이다.

In [ ]:
train = partitions['train']
values = [row['nd_mw'] for row in train]


def summarize(series):
    q1, median, q3 = stats.quantiles(series, n=4, method='inclusive')
    return {'n': len(series), 'mean_mw': stats.mean(series), 'std_mw': stats.stdev(series),
            'min_mw': min(series), 'p25_mw': q1, 'median_mw': median,
            'p75_mw': q3, 'max_mw': max(series)}


nd = summarize(values)
spread = nd['p75_mw'] - nd['p25_mw']
lower = nd['p25_mw'] - 1.5 * spread
upper = nd['p75_mw'] + 1.5 * spread
high = [value > upper for value in values]
high_run_lengths = []
run = 0
for above in [*high, False]:
    if above:
        run += 1
    elif run:
        high_run_lengths.append(run)
        run = 0

show_table(['통계량', '값'], [(label, f'{value:,.2f}') for label, value in nd.items()])
print(f'IQR 검사 범위: {lower:,.1f}–{upper:,.1f} MW')
print(f'하한 미만 {sum(value < lower for value in values)}개 / 상한 초과 {sum(high)}개')
print(f'상한 초과 구간 {len(high_run_lengths)}개 / 고립된 관측 {high_run_lengths.count(1)}개 / 최장 {max(high_run_lengths, default=0)}구간')

extremes = [values.index(min(values)), values.index(max(values))]
show_table(['영국 현지 시각', '직전(MW)', '관측(MW)', '직후(MW)'], [
    (train[i]['timestamp_local'], values[i-1], values[i], values[i+1]) for i in extremes
])

### 4.1 평일·주말과 월별 수요

평일·주말의 관측 건수를 함께 보고, 현지 30분 시간대별 평균과 현지 정산 월별 평균을 계산한다.

In [ ]:
by_day = defaultdict(list)
by_slot = defaultdict(list)
by_month = defaultdict(list)
for row in train:
    value = row['nd_mw']
    by_day[row['is_weekend']].append(value)
    by_slot[row['is_weekend'], row['local_half_hour']].append(value)
    by_month[row['settlement_date'][:7]].append(value)

profiles = {key: stats.mean(series) for key, series in sorted(by_slot.items())}
monthly = {key: stats.mean(series) for key, series in sorted(by_month.items())}
group_rows = []
for group, group_name in [(0, '평일'), (1, '주말')]:
    series = by_day[group]
    peak = max(range(48), key=lambda slot: profiles[group, slot])
    group_rows.append((group_name, f'{len(series):,}', f'{100*len(series)/len(values):.2f}%',
                       f'{stats.mean(series):,.1f}', f'{stats.median(series):,.1f}',
                       f'{peak//2:02d}:{30*(peak%2):02d}', f'{profiles[group, peak]:,.1f}'))

show_table(['집단', '관측 수', '비중', '평균(MW)', '중앙값(MW)', '최고 평균 시간', '최고 평균(MW)'], group_rows)
show_table(['정산 월', '관측 수', '평균(MW)'],
           [(month, f'{len(by_month[month]):,}', f'{mean:,.1f}') for month, mean in monthly.items()])

### 4.2 시차별 관계와 급변 구간

7일 전까지 값이 존재하는 동일한 시점을 대상으로 피어슨 상관과 평균 절대 차이를 비교한다. 24시간은 UTC 기준 경과 시간이다. 급변 구간은 전체 학습 구간의 30분 절대 변화량에서 99백분위 기준을 사용해 확인한다.

In [ ]:
LAGS = {1: '30 min', 2: '1 hour', 12: '6 hours', 24: '12 hours',
        48: '24 hours', 96: '48 hours', 336: '7 days'}
start = max(LAGS)
actual = values[start:]
lags = []
for steps, label in LAGS.items():
    previous = values[start-steps:len(values)-steps]
    lags.append((steps, label, len(actual), stats.correlation(actual, previous),
                 stats.mean(abs(now - then) for now, then in zip(actual, previous))))
show_table(['시차(구간)', '경과 시간', '비교 건수', 'Pearson r', '평균 절대 차이(MW)'],
           [(steps, label, f'{n:,}', f'{r:.6f}', f'{gap:,.1f}') for steps, label, n, r, gap in lags])

changes = [b-a for a, b in zip(values, values[1:])]
abs_changes = [abs(change) for change in changes]
threshold = stats.quantiles(abs_changes, n=100, method='inclusive')[-1]
tail = [change for change in abs_changes if change >= threshold]
show_table(['30분 절대 변화량 검사', '결과'], [
    ('전체 비교 건수', f'{len(changes):,}'), ('중앙값(MW)', f'{stats.median(abs_changes):,.1f}'),
    ('평균(MW)', f'{stats.mean(abs_changes):,.1f}'), ('99백분위(MW)', f'{threshold:,.1f}'),
    ('기준 이상 건수', f'{len(tail):,}'), ('기준 이상 평균(MW)', f'{stats.mean(tail):,.1f}'),
])
show_table(['영국 현지 시각', '직전(MW)', '현재(MW)', '변화량(MW)'], [
    (train[i+1]['timestamp_local'], values[i], values[i+1], changes[i])
    for i in (changes.index(max(changes)), changes.index(min(changes)))
])

### 4.3 그래프

아래의 시각화는 셀에서 계산한 값을 SVG로 표시한다. 분포, 평일·주말 비교, 월별 흐름, 시차별 관계를 차례로 확인할 수 있다.

In [ ]:
BLUE, TEAL, GRAY = '#245783', '#168575', '#506478'


def plot(title, y_label, x_label, x_range, y_range, x_ticks, y_ticks, lines=(), bars=()):
    """Draw a small SVG with axes, line series or bars."""
    left, right, top, bottom = 76, 728, 66, 303
    x_min, x_max = x_range
    y_min, y_max = y_range

    def px(value):
        return left + (value - x_min) / (x_max - x_min) * (right - left)

    def py(value):
        return bottom - (value - y_min) / (y_max - y_min) * (bottom - top)

    def label(x, y, text, anchor='start', size=12, color=GRAY):
        return (f'<text x="{x}" y="{y}" font-size="{size}" fill="{color}" '
                f'text-anchor="{anchor}">{escape(str(text))}</text>')

    svg = ['<svg xmlns="http://www.w3.org/2000/svg" width="760" height="365" '
           'viewBox="0 0 760 365" font-family="sans-serif">',
           '<rect width="760" height="365" fill="white"/>',
           label(22, 31, title, size=21, color=BLUE), label(left, 54, y_label)]
    for tick, text in y_ticks:
        pos = py(tick)
        svg.extend((f'<line x1="{left}" x2="{right}" y1="{pos}" y2="{pos}" stroke="#dfe8ef"/>',
                    label(left-7, pos+4, text, 'end', 11)))
    for tick, text in x_ticks:
        svg.append(label(px(tick), bottom+18, text, 'middle', 10))
    svg.append(label((left+right)/2, bottom+42, x_label, 'middle'))
    for center, value, width in bars:
        y = min(py(0), py(value))
        h = abs(py(0) - py(value))
        w = px(center+width/2) - px(center-width/2)
        svg.append(f'<rect x="{px(center)-w/2}" y="{y}" width="{w}" height="{h}" fill="{TEAL}"/>')
    for points, name, color in lines:
        coords = ' '.join(f'{px(x):.2f},{py(y):.2f}' for x, y in points)
        svg.append(f'<polyline points="{coords}" fill="none" stroke="{color}" stroke-width="2.5"/>')
    for i, (_, name, color) in enumerate(lines):
        if name:
            svg.append(label(545+i*90, 54, name, color=color))
    display(SVG(data=''.join(svg) + '</svg>'))

In [ ]:
bins = Counter(int(value // 1000) * 1000 for value in values)
first, last = min(bins), max(bins) + 1000
ceiling = ((max(bins.values()) + 499) // 500) * 500
plot('30-minute electricity demand distribution', 'Observations', 'National Demand (MW)',
     (first, last), (0, ceiling), [(v, f'{v:,}') for v in range(15000, 45001, 5000)],
     [(v, f'{v:,}') for v in range(0, ceiling+1, 500)],
     bars=[(edge+500, bins[edge], 920) for edge in range(first, last, 1000)])

plot('Daily demand by day type', 'Mean ND (MW)', 'Local time in Great Britain',
     (0, 47), (15000, 40000), [(i*6, f'{i*3:02d}:00') for i in range(8)],
     [(v, f'{v:,}') for v in range(15000, 40001, 5000)],
     lines=[([(slot, profiles[group, slot]) for slot in range(48)], name, color)
            for group, name, color in [(0, 'Weekday', BLUE), (1, 'Weekend', TEAL)]])

In [ ]:
plot('Monthly mean demand', 'Mean ND (MW)', 'Settlement month',
     (0, len(monthly)-1), (18000, 38000),
     [(i, month) for i, month in enumerate(monthly) if i % 3 == 0 or i == len(monthly)-1],
     [(v, f'{v:,}') for v in range(18000, 38001, 5000)],
     lines=[(list(enumerate(monthly.values())), '', TEAL)])

plot('Current vs. past demand correlation', 'Pearson r', 'Elapsed time before target (UTC)',
     (-0.5, len(lags)-0.5), (-1, 1), [(i, row[1]) for i, row in enumerate(lags)],
     [(v, f'{v:g}') for v in (-1, -.5, 0, .5, 1)],
     bars=[(i, row[3], .42) for i, row in enumerate(lags)])

## 5. 기존 결과와 대조

전처리 보존 행 수, 학습 구간 평균, 시차별 통계, 급변 구간의 건수와 주요 값이 기존 프로젝트 결과와 일치하는지 확인한다.

In [ ]:
assert {name: len(rows) for name, rows in partitions.items()} == {'train': 35040, 'validation': 17568, 'test': 17520}
assert len(set(instants)) == 70128
assert math.isclose(nd['mean_mw'], 26337.01709474886, abs_tol=1e-8)
assert sum(high) == 177
assert len(high_run_lengths) == 42
assert len(tail) == 352
assert threshold == 2661.0
assert math.isclose(stats.mean(tail), 2934.369318181818, abs_tol=1e-8)
for (_, _, n, r, gap), (expected_r, expected_gap) in zip(lags, [(0.9891687652071278, 666.5190179806362), (0.9596288303684039, 1291.3397879207007), (0.3555847759620019, 5629.83872176118), (0.16011198533562226, 6499.654477869986), (0.9084025225191423, 1832.722366297833), (0.828840425785232, 2643.6321749654217), (0.8993791037122949, 2018.0858402489628)]):
    assert n == 34704
    assert math.isclose(r, expected_r, abs_tol=1e-10)
    assert math.isclose(gap, expected_gap, abs_tol=1e-8)
print('기존 전처리·EDA 결과와 일치합니다.')